# Explore Wikidata in Python

This notebook helps you interactively inspect what information exists in **Wikidata** for any term or entity.

## What is Wikidata?
Wikidata is an open, collaboratively edited knowledge base of structured data. It stores entities (items) such as people, places, organizations, concepts, and events, along with machine-readable statements about them.

## Key concepts
- **Entity / Item**: A thing in Wikidata, usually identified by IDs like `Q42` (items) and `P31` (properties).
- **Label**: Human-readable name of an entity (for example, "United States").
- **Description**: Short disambiguating text (for example, "country in North America").
- **Alias**: Alternative names, spellings, abbreviations, or synonyms.
- **Statement (Claim)**: A property-value assertion about an entity (for example, `P31` = "instance of").
- **Property**: The relation type used in statements (for example, `P17` = country, `P569` = date of birth).

## APIs used in this notebook
Wikidata can be explored through:
- **MediaWiki Action API** (`https://www.wikidata.org/w/api.php`)
  - `wbsearchentities`: search entities by term
  - `wbgetentities`: fetch detailed entity data
- **Wikidata Query Service** (`https://query.wikidata.org/sparql`) for SPARQL queries

The notebook focuses on readable outputs for exploration and beginner-friendly usage.

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
# Setup: import libraries, install missing packages if needed, and define API endpoints.

import sys
import subprocess
import importlib
import json

def ensure_package(package_name):
    """Install package via pip if not available."""
    try:
        importlib.import_module(package_name)
        print(f"Package '{package_name}' is available.")
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Required third-party packages for this notebook.
for pkg in ["requests", "pandas"]:
    ensure_package(pkg)

import requests
import pandas as pd

pd.set_option("display.max_colwidth", None)
from IPython.display import display, Markdown
from wikidata_utils import extract_best_wikipedia_title, fetch_detailed_descriptions_for_entities, fetch_wikipedia_intro

# Base endpoints (official Wikidata-compatible patterns).
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
WDQS_URL = "https://query.wikidata.org/sparql"

# Good practice: identify your client in requests.
DEFAULT_HEADERS = {
    "User-Agent": "WikidataExplorerNotebook/1.0 (https://www.wikidata.org/)"
}

print("Setup complete.")
print("Wikidata API:", WIKIDATA_API_URL)
print("WDQS endpoint:", WDQS_URL)

Package 'requests' is available.
Package 'pandas' is available.
Setup complete.
Wikidata API: https://www.wikidata.org/w/api.php
WDQS endpoint: https://query.wikidata.org/sparql


## Search Wikidata Entities
Use `wbsearchentities` to find candidate entities for a word or phrase.

Endpoint pattern:
- `action=wbsearchentities`
- `search=<term>`
- `language=<lang>`
- `limit=<n>`

Use `include_detailed_description=True` to append a richer natural-language summary
from the linked Wikipedia lead section.

In [3]:
# Search functions: robust API call + DataFrame output.

def _safe_get_json(url, params=None, headers=None, timeout=30):
    """Safely call an HTTP JSON endpoint and return parsed JSON (or empty dict on failure)."""
    try:
        response = requests.get(url, params=params, headers=headers or DEFAULT_HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
        return {}
    except ValueError as exc:
        print(f"Invalid JSON response: {exc}")
        return {}

def _coerce_aliases_for_search(value):
    """Convert aliases from wbsearchentities result into a clean comma-separated string."""
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v) for v in value if v is not None)
    if isinstance(value, str):
        return value
    return str(value)

def _series_casefold_equals(series, target):
    target_casefold = str(target).casefold()
    return series.map(lambda value: str(value).casefold() == target_casefold if value is not None else False)

def _series_casefold_contains(series, target):
    target_casefold = str(target).casefold()
    return series.map(lambda value: target_casefold in str(value).casefold() if value is not None else False)

def _series_non_empty_mask(series):
    return series.map(lambda value: bool(str(value).strip()) if value is not None else False)

def _series_exclude_name_descriptions(series):
    blocked_phrases = ("family name", "given name")
    return series.map(
        lambda value: not any(phrase in str(value).casefold() for phrase in blocked_phrases)
        if value is not None else True
    )

def search_wikidata(
    term,
    language="en",
    limit=10,
    exact_match_text=False,
    include_detailed_description=False,
    drop_missing_detailed_description=False,
    detailed_description_sentences=3,
    filter_name=True,
    label_contains_text=True,
):
    """
    Search Wikidata entities using wbsearchentities and return a clean DataFrame.

    If exact_match_text is True, only keep rows whose match_text exactly matches
    the input term, ignoring case.

    If label_contains_text is True, only keep rows whose label contains
    the input term, ignoring case.

    If filter_name is True, remove rows whose description contains the
    exact phrases "family name" or "given name".

    If include_detailed_description is True, add a Wikipedia-backed
    detailed_description column for richer natural-language context.

    If drop_missing_detailed_description is True, remove rows whose
    detailed_description is missing after the Wikipedia lookup.

    Columns include: id, label, description, match_text, aliases, concepturi,
    and optionally detailed_description.
    """
    if not isinstance(term, str) or not term.strip():
        print("Please provide a non-empty search term.")
        columns = ["id", "label", "description", "match_text", "aliases", "concepturi"]
        if include_detailed_description:
            columns.insert(3, "detailed_description")
        return pd.DataFrame(columns=columns)

    normalized_term = term.strip()

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": normalized_term,
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")

        row = {
            "id": item.get("id", ""),
            "label": item.get("label", ""),
            "description": item.get("description", ""),
            "match_text": match_text,
            "aliases": _coerce_aliases_for_search(item.get("aliases")),
            "concepturi": item.get("concepturi", ""),
        }
        rows.append(row)

    df = pd.DataFrame(rows, columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    if filter_name and not df.empty:
        df = df[_series_exclude_name_descriptions(df["description"])].reset_index(drop=True)

    if label_contains_text and not df.empty:
        df = df[_series_casefold_contains(df["label"], normalized_term)].reset_index(drop=True)

    if exact_match_text and not df.empty:
        df = df[_series_casefold_equals(df["match_text"], normalized_term)].reset_index(drop=True)

    if include_detailed_description:
        detailed_descriptions = {}
        if not df.empty:
            detailed_descriptions = fetch_detailed_descriptions_for_entities(
                df["id"].tolist(),
                language=language,
                headers=DEFAULT_HEADERS,
                timeout=30,
                sentences=detailed_description_sentences,
            )
        df.insert(
            df.columns.get_loc("description") + 1,
            "detailed_description",
            [detailed_descriptions.get(entity_id, "") for entity_id in df["id"]],
        )
        if drop_missing_detailed_description:
            df = df[_series_non_empty_mask(df["detailed_description"])].reset_index(drop=True)

    if df.empty:
        print(f"No search results for: {term!r}")
    return df

## Entity Detail Inspection
Use `wbgetentities` to retrieve details for a known entity ID (such as `Q30` or `Q312`).

Endpoint pattern:
- `action=wbgetentities`
- `ids=<entity_id>`
- `props=labels|descriptions|aliases|sitelinks|claims`

This section formats claims safely into readable text, with fallback handling for unexpected data shapes.
It can also show a richer summary pulled from the linked Wikipedia page.

In [4]:
# Entity detail utilities and robust claim decoding.

def _extract_lang_value(lang_map, language="en"):
    """Get value for preferred language with fallback to first available."""
    if not isinstance(lang_map, dict) or not lang_map:
        return ""
    if language in lang_map and isinstance(lang_map[language], dict):
        return lang_map[language].get("value", "")
    first = next(iter(lang_map.values()), {})
    return first.get("value", "") if isinstance(first, dict) else ""

def _extract_aliases(alias_map, language="en"):
    """Get alias list for language with fallback to any language."""
    if not isinstance(alias_map, dict) or not alias_map:
        return []

    entries = alias_map.get(language)
    if not entries and alias_map:
        entries = next(iter(alias_map.values()), [])

    if not isinstance(entries, list):
        return []

    aliases = []
    for ent in entries:
        if isinstance(ent, dict):
            val = ent.get("value")
            if val:
                aliases.append(val)
    return aliases

def _decode_datavalue(datavalue):
    """Decode common Wikidata datavalue shapes into readable text."""
    if not isinstance(datavalue, dict):
        return str(datavalue)

    dtype = datavalue.get("type")
    value = datavalue.get("value")

    try:
        # wikibase-item / wikibase-property references
        if dtype == "wikibase-entityid" and isinstance(value, dict):
            entity_id = value.get("id")
            entity_type = value.get("entity-type", "entity")
            if entity_id:
                return f"{entity_id} ({entity_type})"
            if value.get("numeric-id") is not None:
                prefix = "Q" if entity_type == "item" else "P"
                return f"{prefix}{value['numeric-id']} ({entity_type})"
            return json.dumps(value, ensure_ascii=False)

        # plain strings
        if dtype == "string":
            return str(value)

        # monolingual text
        if dtype == "monolingualtext" and isinstance(value, dict):
            return f"{value.get('text', '')} [{value.get('language', '')}]"

        # time value
        if dtype == "time" and isinstance(value, dict):
            time_str = value.get("time", "")
            precision = value.get("precision", "")
            return f"{time_str} (precision={precision})"

        # quantity value
        if dtype == "quantity" and isinstance(value, dict):
            amount = value.get("amount", "")
            unit = value.get("unit", "")
            if isinstance(unit, str) and unit.startswith("http://www.wikidata.org/entity/"):
                unit = unit.rsplit("/", 1)[-1]
            return f"{amount} (unit={unit if unit else '1'})"

        # coordinates
        if dtype == "globecoordinate" and isinstance(value, dict):
            lat = value.get("latitude")
            lon = value.get("longitude")
            prec = value.get("precision")
            return f"lat={lat}, lon={lon}, precision={prec}"

        # Common fallback for other scalar types
        if isinstance(value, (str, int, float, bool)) or value is None:
            return str(value)

        # Safe fallback for unhandled complex types
        return json.dumps(value, ensure_ascii=False)

    except Exception:
        # Last-resort defensive fallback
        return str(datavalue)

def _decode_claim_value(claim):
    """Extract a readable value from one claim object."""
    if not isinstance(claim, dict):
        return "(invalid claim shape)"

    mainsnak = claim.get("mainsnak", {})
    if not isinstance(mainsnak, dict):
        return "(missing mainsnak)"

    snaktype = mainsnak.get("snaktype", "value")
    if snaktype != "value":
        return f"({snaktype})"

    datavalue = mainsnak.get("datavalue")
    if datavalue is None:
        return "(no datavalue)"

    return _decode_datavalue(datavalue)

def _get_entity_raw(entity_id, language="en"):
    """Fetch raw entity JSON from wbgetentities."""
    if not isinstance(entity_id, str) or not entity_id.strip():
        print("Please provide a valid entity ID like 'Q30'.")
        return None

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": entity_id.strip(),
        "languages": language,
        "props": "labels|descriptions|aliases|sitelinks|claims",
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    entities = data.get("entities", {}) if isinstance(data, dict) else {}
    entity = entities.get(entity_id.strip()) if isinstance(entities, dict) else None

    if not entity or entity.get("missing") == "":
        print(f"Entity '{entity_id}' not found or missing.")
        return None

    return entity

def get_entity_details(entity_id, language="en", include_detailed_description=True, detailed_description_sentences=3):
    """
    Fetch and display key entity details in readable form.
    Returns a structured dict for optional downstream use.
    """
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return None

    eid = entity.get("id", entity_id)
    label = _extract_lang_value(entity.get("labels", {}), language=language)
    description = _extract_lang_value(entity.get("descriptions", {}), language=language)
    aliases = _extract_aliases(entity.get("aliases", {}), language=language)

    sitelinks = entity.get("sitelinks", {}) if isinstance(entity.get("sitelinks"), dict) else {}
    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}
    detailed_description = ""
    wikipedia_title = extract_best_wikipedia_title(entity, language=language)
    if include_detailed_description and wikipedia_title:
        detailed_description = fetch_wikipedia_intro(
            wikipedia_title,
            language=language,
            headers=DEFAULT_HEADERS,
            timeout=30,
            sentences=detailed_description_sentences,
        )

    print("=" * 100)
    print(f"Entity ID   : {eid}")
    print(f"Label       : {label if label else '(missing)'}")
    print(f"Description : {description if description else '(missing)'}")
    print(f"Aliases     : {', '.join(aliases) if aliases else '(none)'}")
    if include_detailed_description:
        print(f"Wikipedia   : {wikipedia_title if wikipedia_title else '(missing sitelink)'}")
        print(f"Summary     : {detailed_description if detailed_description else '(missing)'}")
    print(f"Sitelinks   : {len(sitelinks)}")
    print(f"Properties with claims: {len(claims)}")
    print("=" * 100)

    # Display sitelinks in tabular form for readability.
    if sitelinks:
        site_rows = []
        for site, info in sitelinks.items():
            if not isinstance(info, dict):
                continue
            site_rows.append({
                "site": site,
                "title": info.get("title", ""),
                "url": info.get("url", ""),
            })
        if site_rows:
            print("\nSitelinks (first 20):")
            display(pd.DataFrame(site_rows).head(20))
    else:
        print("\nSitelinks: (none)")

    # Display simplified claims grouped by property ID.
    print("\nClaims grouped by property (simplified values):")
    if not claims:
        print("  (no claims)")
    else:
        for prop_id, claim_list in claims.items():
            print(f"\n  {prop_id}:")
            if not isinstance(claim_list, list) or not claim_list:
                print("    - (none)")
                continue
            for claim in claim_list[:10]:
                print(f"    - {_decode_claim_value(claim)}")
            if len(claim_list) > 10:
                print(f"    ... ({len(claim_list) - 10} more values)")

    return {
        "id": eid,
        "label": label,
        "description": description,
        "aliases": aliases,
        "wikipedia_title": wikipedia_title,
        "detailed_description": detailed_description,
        "sitelinks": sitelinks,
        "claims": claims,
    }

## Helper Functions
Convenience wrappers for common interactive exploration tasks.

In [5]:
# Helper functions requested in the specification.

def print_entity_summary(entity_id, language="en"):
    """Print a compact summary: ID, label, description, and counts."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    label = _extract_lang_value(entity.get("labels", {}), language=language)
    description = _extract_lang_value(entity.get("descriptions", {}), language=language)
    aliases = _extract_aliases(entity.get("aliases", {}), language=language)
    sitelinks = entity.get("sitelinks", {}) if isinstance(entity.get("sitelinks"), dict) else {}
    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}

    print("=" * 80)
    print(f"ID          : {entity.get('id', entity_id)}")
    print(f"Label       : {label if label else '(missing)'}")
    print(f"Description : {description if description else '(missing)'}")
    print(f"Alias count : {len(aliases)}")
    print(f"Sitelinks   : {len(sitelinks)}")
    print(f"Claim props : {len(claims)}")
    print("=" * 80)

def show_aliases(entity_id, language="en"):
    """Show aliases for an entity in selected language (with fallback)."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    aliases = _extract_aliases(entity.get("aliases", {}), language=language)
    print(f"Aliases for {entity.get('id', entity_id)} ({language}):")
    if not aliases:
        print("  (none)")
        return

    for a in aliases:
        print(f"  - {a}")

def show_claims(entity_id, language="en", max_properties=20, max_values_per_property=5):
    """Show simplified claims grouped by property, with configurable truncation."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}
    if not claims:
        print("No claims found.")
        return

    prop_ids = list(claims.keys())
    print(f"Showing up to {max_properties} properties for {entity_id}:")

    for prop_id in prop_ids[:max_properties]:
        values = claims.get(prop_id, [])
        print(f"\n{prop_id}:")

        if not isinstance(values, list) or not values:
            print("  - (none)")
            continue

        for claim in values[:max_values_per_property]:
            print(f"  - {_decode_claim_value(claim)}")

        if len(values) > max_values_per_property:
            print(f"  ... ({len(values) - max_values_per_property} more values)")

    if len(prop_ids) > max_properties:
        print(f"\n... ({len(prop_ids) - max_properties} more properties not shown)")

## Optional: Wikidata Query Service (SPARQL)
This function runs SPARQL queries against `https://query.wikidata.org/sparql` and returns a DataFrame.

In [6]:
# Optional SPARQL support for structured graph queries.

def run_wikidata_sparql(query, timeout=60):
    """Run SPARQL query on WDQS and return results as a DataFrame."""
    if not isinstance(query, str) or not query.strip():
        print("Please provide a non-empty SPARQL query string.")
        return pd.DataFrame()

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": DEFAULT_HEADERS["User-Agent"],
    }

    try:
        response = requests.get(
            WDQS_URL,
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=timeout,
        )
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as exc:
        print(f"SPARQL request failed: {exc}")
        return pd.DataFrame()
    except ValueError as exc:
        print(f"Invalid SPARQL JSON response: {exc}")
        return pd.DataFrame()

    bindings = data.get("results", {}).get("bindings", [])
    if not bindings:
        print("SPARQL returned no rows.")
        return pd.DataFrame()

    rows = []
    for b in bindings:
        row = {}
        for var, payload in b.items():
            if isinstance(payload, dict):
                row[var] = payload.get("value", "")
            else:
                row[var] = str(payload)
        rows.append(row)

    return pd.DataFrame(rows)

## Demos: Search Terms
Try these terms and inspect candidate entities.

In [7]:
# Demo 1: Apple
df_apple = search_wikidata("madonna", limit=10)
display(df_apple)

,id,label,description,match_text,aliases,concepturi
0,Q1744,Madonna,"American singer, songwriter, and actress (born 1958)",Madonna,,http://www.wikidata.org/entity/Q1744
1,Q345,Mary,mother of Jesus Christ,Madonna,Madonna,http://www.wikidata.org/entity/Q345
2,Q9309699,Madonna and Child,artistic depiction of Mary with her child Jesus,Madonna and Child,,http://www.wikidata.org/entity/Q9309699
3,Q6728364,Madonna,2011 single by Secret K-Pop,Madonna,,http://www.wikidata.org/entity/Q6728364
4,Q926743,Madonna,"artistic representation of Mary, either alone or with her child Jesus",Madonna,,http://www.wikidata.org/entity/Q926743
5,Q18890421,Madonna,painting by Edvard Munch,Madonna,,http://www.wikidata.org/entity/Q18890421
6,Q137393279,Madonna,painting by Carlo Dolci,Madonna,,http://www.wikidata.org/entity/Q137393279
7,Q131399734,Madonna,"rock pinnacle in the Allgäu Alps at the border of Bavaria, Germany, and Tyrol, Austria",Madonna,,http://www.wikidata.org/entity/Q131399734
8,Q136724019,Madonna,painting by Carl Gottlieb Peschel,Madonna,,http://www.wikidata.org/entity/Q136724019


In [26]:
# Demo 2: United States
df_us = search_wikidata("battle", include_detailed_description=True,limit=10)
display(df_us)

,id,label,description,detailed_description,match_text,aliases,concepturi
0,Q737593,Battle,"town and civil parish in the local government district of Rother in East Sussex, England","Battle is a town and civil parish in the district of Rother in East Sussex, England. It lies 50 miles (80 km) south-east of London, 27 miles (43 km) east of Brighton and 20 miles (32 km) east of Lewes. Hastings is to the south-east and Bexhill-on-Sea to the south.",Battle,,http://www.wikidata.org/entity/Q737593
1,Q178561,battle,"part of a war which is well defined in duration, area and force commitment","According to one of the possible definitions, a battle is an occurrence of combat in warfare between opposing military units of any number or size. A war usually consists of multiple battles. In general, a battle is a series of military engagements that is well defined in duration, area, and force commitment.",battle,,http://www.wikidata.org/entity/Q178561
2,Q24656443,Battle,"civil parish in East Sussex, England, UK",,Battle,,http://www.wikidata.org/entity/Q24656443
3,Q114811775,Battle,video game,,Battle,,http://www.wikidata.org/entity/Q114811775
4,Q810998,Battle Creek,"city in Calhoun County, Michigan, United States","Battle Creek is a city in northwestern Calhoun County, Michigan, United States, at the confluence of the Kalamazoo and Battle Creek rivers. As of the 2020 census, the city had a total population of 52,731. It is the principal city of the Battle Creek metropolitan statistical area, which encompasses all of Calhoun County.",Battle Creek,,http://www.wikidata.org/entity/Q810998
5,Q66121500,Battle for Dream Island,American independent-animated web series,"Battle for Dream Island (BFDI) is an American animated web series created by twin brothers Cary and Michael Huang. A parody of the game show genre, the series consists of competitions between anthropomorphic objects, with viewers voting for a contestant's elimination. Episodes and related media are posted on their YouTube channel, jacknjellify.",Battle for Dream Island,,http://www.wikidata.org/entity/Q66121500
6,Q105826326,battle,act of struggling to achieve or fight against something,,battle,,http://www.wikidata.org/entity/Q105826326
7,Q33570,Battle of Grunwald,1410 battle between the Teutonic Knights and Poland–Lithuania,"The Battle of Grunwald was fought on 15 July 1410 during the Polish–Lithuanian–Teutonic War. The alliance of the Crown of the Kingdom of Poland and the Grand Duchy of Lithuania, led respectively by King Władysław II Jagiełło (Jogaila), and Grand Duke Vytautas, decisively defeated the Teutonic Order, led by Grand Master Ulrich von Jungingen. Most of the Teutonic Order's leadership was killed or taken prisoner.",Battle of Grunwald,,http://www.wikidata.org/entity/Q33570
8,Q157627,Battle of the Atlantic,1939 longest continuous military campaign in World War II,"The Battle of the Atlantic, the longest-continuous military campaign in World War II, ran from 1939 to the defeat of Nazi Germany in 1945, covering a major part of the naval history of World War II. At its core was the Allied naval blockade of Germany, announced the day after the declaration of war, and Germany's subsequent counterblockade. The campaign peaked from mid-1940 to the end of 1943. The Battle of the Atlantic pitted U-boats and other warships of the German Kriegsmarine (navy) and aircraft of the Luftwaffe (air force) against the Royal Navy, Royal Canadian Navy, United States Navy, and Allied merchant shipping.",Battle of the Atlantic,,http://www.wikidata.org/entity/Q157627


In [15]:
# Demo 3: director
df = search_wikidata("event",limit=10,include_detailed_description=True,detailed_description_sentences=3,drop_missing_detailed_description=True)
display(df)

,id,label,description,detailed_description,match_text,aliases,concepturi
0,Q1259759,miniseries,TV shows or series that have a predetermined number of episodes,"A miniseries or mini-series, sometimes called a limited-run series, is a television program that tells a story in a predetermined, limited number of episodes. Many miniseries can also be referred to, and shown, as a television film in several parts. The term ""serial"" is used in the United Kingdom and in other Commonwealth nations to describe a show that has an ongoing narrative plotline, while ""series"" is used for a set of episodes in a similar way that ""season"" is used in North America.",event series,event series,http://www.wikidata.org/entity/Q1259759
1,Q265158,review,"evaluation of a publication, service, company, piece of hardware, event, exhibition, or performance","A review is an evaluation of a publication, product, service, or company or a critical take on current affairs in literature, politics or culture. In addition to a critical evaluation, the review's author may assign the work a rating to indicate its relative merit. Reviews can apply to a movie, video game, musical composition, book; a piece of hardware like a car, home appliance, or computer; or software such as business software, sales software; or an event or performance, such as a live music concert, play, musical theater show, dance show or art exhibition.",event review,event review,http://www.wikidata.org/entity/Q265158
2,Q10290214,event,"in statistics and probability theory, set of outcomes to which a probability is assigned","In probability theory, an event is a subset of outcomes of an experiment (a subset of the sample space) to which a probability is assigned. A single outcome may be an element of many different events, and different events in an experiment are usually not equally likely, since they may include very different groups of outcomes. An event consisting of only a single outcome is called an elementary event or an atomic event; that is, it is a singleton set.",event,,http://www.wikidata.org/entity/Q10290214
3,Q641226,arena,"enclosed area designed to host sporting events, theater, and musical performances","An arena is a large enclosed venue, often circular or oval-shaped, designed to showcase theatre, musical performances or sporting events. It comprises a large open space surrounded on most or all sides by tiered seating for spectators, and may be covered by a roof. The key feature of an arena is that the event space is the lowest point, allowing maximum visibility.",events centre,events centre,http://www.wikidata.org/entity/Q641226


In [ ]:
%%sql


In [10]:
# Demo 4: OpenAI
df = search_wikidata("director", limit=10, exact_match_text=True)
display(df)

,id,label,description,match_text,aliases,concepturi
0,Q2526255,film director,person who controls the artistic and dramatic aspects of a film production,director,director,http://www.wikidata.org/entity/Q2526255
1,Q3455803,director,director of a creative work,director,,http://www.wikidata.org/entity/Q3455803
2,Q3387717,theatrical director,person overseeing the mounting of a theatre production,director,director,http://www.wikidata.org/entity/Q3387717
3,Q1162163,director,person who leads a particular area of a company or organization,director,,http://www.wikidata.org/entity/Q1162163
4,Q27714113,The Director,journal of the National Association of Directors of Nursing Administration in Long-Term Care,Director,Director,http://www.wikidata.org/entity/Q27714113


## Demos: Inspect One Entity in Depth
Set any entity ID you want to inspect below.

Tip: pick an ID from the search results above (for example, often `Q30` for United States and `Q24210` for Apple Inc.).

In [11]:
# Pick an entity ID and inspect details.
entity_id = "Q30"  # Change this interactively

print_entity_summary(entity_id)
show_aliases(entity_id)
show_claims(entity_id, max_properties=15, max_values_per_property=5)

# Full detail display
_ = get_entity_details(entity_id)

ID          : Q30
Label       : United States
Description : country located primarily in North America
Alias count : 22
Sitelinks   : 424
Claim props : 455
Aliases for Q30 (en):
  - the States
  - the United States of America
  - US of America
  - the US
  - the U.S.
  - the US of A
  - U.S. of America
  - the US of America
  - the USA
  - the U.S.A.
  - the U.S. of A
  - US of A
  - the U.S. of America
  - the United States
  - Merica
  - Murica
  - United States of America
  - U.S.
  - U.S.A.
  - U. S.
  - U. S. A.
  - America
Showing up to 15 properties for Q30:

P2924:
  - 3590227

P1344:
  - Q488 (item)
  - Q280468 (item)
  - Q1088364 (item)
  - Q40949 (item)
  - Q184425 (item)
  ... (27 more values)

P409:
  - 35562417

P1333:
  - lat=24.5442989, lon=-81.8051241, precision=1e-07

P3134:
  - 191

P3479:
  - 9050eaf972828656c88d8704360f0b8e65bee3f0

P3221:
  - destination/united-states

P227:
  - 4078704-7

P1792:
  - Q3919762 (item)

P2852:
  - Q533806 (item)

P2853:
  - Q24288454

,site,title,url
0,abstractwiki,Q30,
1,abwiki,Еиду Америкатәи Аштатқәа,
2,acewiki,Amirika Syarikat,
3,adywiki,Америкэ Штат Зэхэтхэр,
4,afwiki,Verenigde State van Amerika,
5,alswiki,USA,
6,amiwiki,Amilika,
7,amwiki,የተባበሩት የአሜሪካ ግዛቶች,
8,angwiki,Geanedan Ricu America,
9,anpwiki,अमेरिका,



Claims grouped by property (simplified values):

  P2924:
    - 3590227

  P1344:
    - Q488 (item)
    - Q280468 (item)
    - Q1088364 (item)
    - Q40949 (item)
    - Q184425 (item)
    - Q178810 (item)
    - Q155723 (item)
    - Q8740 (item)
    - Q8663 (item)
    - Q169401 (item)
    ... (22 more values)

  P409:
    - 35562417

  P1333:
    - lat=24.5442989, lon=-81.8051241, precision=1e-07

  P3134:
    - 191

  P3479:
    - 9050eaf972828656c88d8704360f0b8e65bee3f0

  P3221:
    - destination/united-states

  P227:
    - 4078704-7

  P1792:
    - Q3919762 (item)

  P2852:
    - Q533806 (item)

  P2853:
    - Q24288454 (item)
    - Q24288456 (item)

  P395:
    - USA

  P2581:
    - 00003341n

  P1943:
    - Usa edcp location map.svg
    - Location map of USA (+HI +AK +Unincorporated territories + DC).svg
    - Map of USA with state names.svg

  P1125:
    - +47.7 (unit=1)
    - +34.6 (unit=1)

  P1465:
    - Q6334989 (item)

  P3219:
    - etats-unis-d-amerique-vue-d-ensemble

 

In [12]:
# Optional SPARQL demo: retrieve a few direct properties for one entity.
# Example uses Q30 (United States). Replace with another entity ID if desired.

selected_entity = "Q30"
sparql_query = f"""
SELECT ?prop ?propLabel ?value ?valueLabel WHERE {{
  wd:{selected_entity} ?p ?value .
  ?prop wikibase:directClaim ?p .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT 25
"""

df_sparql = run_wikidata_sparql(sparql_query)
display(df_sparql)

,value,prop,propLabel,valueLabel
0,http://www.wikidata.org/entity/Q115,http://www.wikidata.org/entity/P530,diplomatic relation,Ethiopia
1,http://www.wikidata.org/entity/Q117,http://www.wikidata.org/entity/P530,diplomatic relation,Ghana
2,http://www.wikidata.org/entity/Q142,http://www.wikidata.org/entity/P530,diplomatic relation,France
3,http://www.wikidata.org/entity/Q145,http://www.wikidata.org/entity/P530,diplomatic relation,United Kingdom
4,http://www.wikidata.org/entity/Q148,http://www.wikidata.org/entity/P530,diplomatic relation,People's Republic of China
5,http://www.wikidata.org/entity/Q155,http://www.wikidata.org/entity/P530,diplomatic relation,Brazil
6,http://www.wikidata.org/entity/Q159,http://www.wikidata.org/entity/P530,diplomatic relation,Russia
7,http://www.wikidata.org/entity/Q183,http://www.wikidata.org/entity/P530,diplomatic relation,Germany
8,http://www.wikidata.org/entity/Q184,http://www.wikidata.org/entity/P530,diplomatic relation,Belarus
9,http://www.wikidata.org/entity/Q189,http://www.wikidata.org/entity/P530,diplomatic relation,Iceland


## RAG_graph Ranking Debug


In [27]:
DEBUG_TERM = ("series")
DEBUG_LIMIT = 5
DEBUG_REQUIRE_DETAILED_DESCRIPTION = True
DEBUG_FILTER_NAME = True
DEBUG_LANGUAGE = "en"


def _series_startswith_lowercase_debug(series):
    return series.map(
        lambda value: (str(value)[0].islower() if value is not None and str(value) else False)
    )


def debug_rag_graph_candidate_ranking(
    term,
    language="en",
    limit=5,
    require_detailed_description=True,
    filter_name=True,
):
    normalized_term = term.strip()
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": normalized_term,
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")
        rows.append(
            {
                "id": item.get("id", ""),
                "label": item.get("label", ""),
                "description": item.get("description", ""),
                "match_text": match_text,
                "aliases": _coerce_aliases_for_search(item.get("aliases")),
                "concepturi": item.get("concepturi", ""),
            }
        )

    raw_df = pd.DataFrame(
        rows,
        columns=["id", "label", "description", "match_text", "aliases", "concepturi"],
    )
    if filter_name and not raw_df.empty:
        raw_df = raw_df[_series_exclude_name_descriptions(raw_df["description"])].reset_index(drop=True)

    ranked_df = raw_df.copy()
    if not ranked_df.empty:
        ranked_df = ranked_df.assign(
            exact_match_rank=_series_casefold_equals(ranked_df["match_text"], normalized_term).astype(int),
            lowercase_initial_rank=_series_startswith_lowercase_debug(ranked_df["match_text"]).astype(int),
        )
        ranked_df = ranked_df.sort_values(
            ["exact_match_rank", "lowercase_initial_rank"],
            ascending=[False, False],
            kind="stable",
        ).reset_index(drop=True)

    if require_detailed_description:
        detailed_descriptions = {}
        if not ranked_df.empty:
            detailed_descriptions = fetch_detailed_descriptions_for_entities(
                ranked_df["id"].tolist(),
                language=language,
                headers=DEFAULT_HEADERS,
                timeout=30,
                sentences=3,
            )
        ranked_df.insert(
            ranked_df.columns.get_loc("description") + 1,
            "detailed_description",
            [detailed_descriptions.get(entity_id, "") for entity_id in ranked_df["id"]],
        )
        ranked_df = ranked_df[_series_non_empty_mask(ranked_df["detailed_description"])].reset_index(drop=True)

    return raw_df, ranked_df


raw_df, rag_ranked_df = debug_rag_graph_candidate_ranking(
    DEBUG_TERM,
    language=DEBUG_LANGUAGE,
    limit=DEBUG_LIMIT,
    require_detailed_description=DEBUG_REQUIRE_DETAILED_DESCRIPTION,
    filter_name=DEBUG_FILTER_NAME,
)

print("RAG_graph.py-style ranked results:")
display(
    rag_ranked_df[
        [
            "id",
            "label",
            "description",
            "match_text",
            "exact_match_rank",
            "lowercase_initial_rank",
            *(["detailed_description"] if "detailed_description" in rag_ranked_df.columns else []),
        ]
    ]
)


RAG_graph.py-style ranked results:


,id,label,description,match_text,exact_match_rank,lowercase_initial_rank,detailed_description
0,Q170198,series,infinite sum,series,1,1,"In mathematics, a series is, roughly speaking, an addition of infinitely many terms, one after the other. The study of series is a major part of calculus and its generalization, mathematical analysis. Series are used in most areas of mathematics, even for studying finite structures in combinatorics through generating functions."
1,Q3464665,television series season,set of episodes produced for a television series,series season,0,1,"A television show, TV program (British English: programme), or simply a TV show, is the general reference to any content produced for viewing on a television set that is transmitted via over-the-air, satellite, and cable, or distributed digitally on streaming platforms. This generally excludes breaking news or advertisements that are aired between shows or between segments of a show. A regularly recurring show is called a television series, and an individual segment of such a series is called an episode."
2,Q578109,television producer,occupation within video production for TV,series producer,0,1,"A television producer is a person who oversees one or more aspects of a television program. Some producers take more of an executive role, in that they conceive new programs and pitch them to the television networks, but upon acceptance they focus on business matters, such as budgets and contracts. Other producers are more involved with the day-to-day workings, participating in activities such as screenwriting, set design, casting, and directing."
3,Q7725310,series of creative works,ordered set of creative works,series of creative works,0,1,"Series fiction refers to a group of independently published works of fiction that are related to one another, usually through similar elements of setting and characters. A common example of series fiction is a book series. Series fiction spans a wide range of genres, and is particularly common in adventure, mystery, romance, fantasy, and science fiction."


## Why Wikidata Is Useful
Wikidata works especially well for:
- **Named entities** (people, organizations, places, products, events)
- **Aliases and variant names** (alternate spellings, abbreviations, multilingual names)
- **Entity disambiguation** (same surface word, different meanings/entities)
- **Structured knowledge** (properties and typed values that are machine-readable)

This makes Wikidata a strong resource for tasks like entity linking, search enrichment, abbreviation expansion, and building knowledge-aware NLP pipelines.